# Statement analytics

**Purpose:** Use `finstack_quant.statements_analytics` to stress-test, compare, explain, and score forecasts on a statement model built with `ModelBuilder`.

**Prerequisites:** Work through `04_statement_modeling/statement_modeling.ipynb` so you are comfortable building periods, value nodes, compute nodes, and evaluating with `Evaluator`.


## Analytics on statement models

A financial model is a directed graph of periods and nodes. Once you can evaluate it, analytics answers adjacent questions: *What if drivers move?* *How does a new plan differ from the old one?* *What revenue clears a target EBITDA?* *Which scenarios matter?* *Where did this number come from?* *Was the forecast any good?*

The Python bindings expose these workflows as small functions that take the typed model, its evaluated results and typed configurations (or simple numeric vectors) and return typed Python values—ideal for notebooks, dashboards, and automated reports.


## Shared baseline: a compact P&L

We build one demo model, `spec`, and evaluate it to `result`. Later calls reuse these two objects so each analytics API stays a pure function of model + configuration. `spec.to_json()` and `result.to_json()` give their JSON documents; `FinancialModelSpec.from_json` and `StatementResult.from_json` load such text back into the objects the analytics functions take.


In [ ]:
import json

from finstack_quant.statements import Evaluator, ModelBuilder

b = ModelBuilder("analytics-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.source_snapshot("teaching-close-2025Q2")
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)])
b.compute("ebitda", "gross_profit - opex")
for node in ("revenue", "cogs", "opex"):
    b.availability_dates(node, [("2025Q1", "2025-04-01"), ("2025Q2", "2025-07-01")])
spec = b.build()
model_json = spec.to_json()
result = Evaluator().evaluate(spec)
eval_json = result.to_json()

print("Model id:", json.loads(model_json)["id"])
print("EBITDA 2025Q1 (evaluated):", json.loads(eval_json)["cells"]["ebitda"]["2025Q1"]["value"])

## Sensitivity and tornado views

`run_sensitivity` perturbs explicit driver nodes and records how target metrics respond. `generate_tornado_entries` returns a ranked list of typed `TornadoEntry` objects for one metric (and optional period) — useful for plotting.

**Forecast-only semantics:** `run_sensitivity` may only override forecast periods. In this demo, `actuals_until="2025Q2"`, so sensitivities should target `2025Q3` or `2025Q4`.

**Perturbation semantics:** Each entry in `perturbations` is an **absolute replacement value** for the node in that period. The `base_value` field records the original level and is used by `generate_tornado_entries` to classify perturbations as downside (below base) or upside (above base). For example, with `base_value=115` and `perturbations=[95, ..., 135]`, each value directly replaces revenue in the selected forecast period.

In [ ]:
from finstack_quant.statements_analytics import (
    SensitivityConfig,
    generate_tornado_entries,
    run_sensitivity,
)

sensitivity_config = SensitivityConfig(
    mode="diagonal",
    parameters=[
        (
            "revenue",
            "2025Q3",
            115.0,
            [95.0, 105.0, 110.0, 115.0, 120.0, 125.0, 135.0],
        )
    ],
    target_metrics=["gross_profit", "ebitda"],
)
sens_result = run_sensitivity(spec, sensitivity_config)

print(f"Sensitivity: {len(sens_result)} scenarios across {sens_result.target_metrics}")
for scenario_index in range(len(sens_result)):
    revenue_q3 = sens_result.get_parameter_value(scenario_index, "revenue@2025Q3")
    ebitda_q3 = sens_result.get_value(scenario_index, "ebitda", "2025Q3")
    print(f"  revenue@Q3 = {revenue_q3:6.0f} -> EBITDA@Q3 = {ebitda_q3:8.1f}")

tornado = generate_tornado_entries(sens_result, "ebitda", "2025Q3")
tornado_values = [
    {"parameter_id": entry.parameter_id, "downside": entry.downside, "upside": entry.upside}
    for entry in tornado
]
print("\nTornado entries:", tornado_values)

## Variance between two evaluated models

`run_variance` compares two **evaluated** `StatementResult` objects (baseline vs comparison) for selected metrics and periods. Build a second `ModelBuilder`, evaluate it, and pass both results plus a `VarianceConfig`.


In [ ]:
from finstack_quant.statements_analytics import VarianceConfig, run_variance

b2 = ModelBuilder("comparison")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("revenue", [("2025Q1", 105.0), ("2025Q2", 115.0), ("2025Q3", 120.0), ("2025Q4", 125.0)])
b2.value("cogs", [("2025Q1", 62.0), ("2025Q2", 67.0), ("2025Q3", 70.0), ("2025Q4", 74.0)])
b2.compute("gross_profit", "revenue - cogs")
b2.value("opex", [("2025Q1", 21.0), ("2025Q2", 23.0), ("2025Q3", 24.0), ("2025Q4", 25.0)])
b2.compute("ebitda", "gross_profit - opex")
comp_result = Evaluator().evaluate(b2.build())

variance_config = VarianceConfig(
    "base",
    "upside",
    ["gross_profit", "ebitda"],
    ["2025Q1", "2025Q2"],
)
variance = run_variance(result, comp_result, variance_config)
print(f"Variance: {variance.baseline_label} vs {variance.comparison_label}")
for row in variance.rows:
    print(
        f"  {row.period} {row.metric:15s}  base={row.baseline:6.1f}  cmp={row.comparison:6.1f}  "
        f"abs={row.abs_var:+.1f}  pct={row.pct_var:+.1%}"
    )

## Goal seek

Solve a forecast assumption on a fixed information cutoff and valuation date.
The reporting target has its own absolute tolerance. The driver stays inside
the stated bracket. A fresh evaluation verifies the target at the published
driver value and the input model remains immutable.


In [ ]:
from finstack_quant.statements_analytics import GoalSeekSpec, goal_seek

goal_spec = GoalSeekSpec(
    driver={"node": "revenue", "period": "2025Q3"},
    target={"node": "ebitda", "period": "2025Q3"},
    target_value=25.0, target_tolerance=0.000001,
    driver_tolerance=1e-10, max_iterations=100,
    search={"kind": "bracket", "lower": 100.0, "upper": 140.0},
    as_of="2025-07-01", observed_through="2025-07-01", update_model=False,
)
gs_result = goal_seek(spec, goal_spec)
assert abs(gs_result.achieved_value - 25.0) <= gs_result.target_tolerance
assert gs_result.model is None
print("Solved revenue (2025Q3):", gs_result.solved_value)


## Scenario set evaluation

`evaluate_scenario_set` runs every named scenario against the same base model. Each scenario supplies `overrides` as **node id → scalar**; those scalars replace values only in **forecast periods** (periods after `actuals_until`). Actual periods are never overridden — they retain their original values regardless of the override.

In this model, `actuals_until="2025Q2"`, so overrides only affect Q3 and Q4. Below, upside and downside shift `revenue` in the forecast window.


In [ ]:
from finstack_quant.statements_analytics import ScenarioSet, evaluate_scenario_set

scenario_set = ScenarioSet({
    "upside": {"revenue": 130.0},
    "downside": {"revenue": 100.0},
})
scenario_results = evaluate_scenario_set(spec, scenario_set)
for name in scenario_results.names:
    scenario_result = scenario_results.get(name)
    print(f"--- {name} ---")
    for period in ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]:
        revenue = scenario_result.get("revenue", period)
        ebitda = scenario_result.get("ebitda", period)
        print(f"  {period}  revenue={revenue:6.1f}  ebitda={ebitda:5.1f}")
print("\nNote: Q1–Q2 (actuals) retain original values; Q3–Q4 (forecast) use overrides.")

## Dependencies and formula explanation

`DependencyTracer.dependency_tree_text()` returns an ASCII tree (`dependency_tree()` the typed `DependencyTree`) of upstream nodes. `DependencyTracer.direct_dependencies()` lists immediate inputs. `explain_formula` returns a typed `Explanation` with the resolved breakdown for one node and period—use `to_json()` for a stable string view.


In [ ]:
from finstack_quant.statements_analytics import DependencyTracer, explain_formula

deps = DependencyTracer(spec).dependency_tree_text("ebitda")
print("Full dependency tree:")
print(deps)

direct = DependencyTracer(spec).direct_dependencies("ebitda")
print("Direct deps:", direct)

explanation = explain_formula(spec, result, "ebitda", "2025Q1")
print("Formula explanation (JSON):", json.dumps(json.loads(explanation.to_json()), indent=2))


## Backtest forecast accuracy

`backtest_forecast` expects parallel sequences of actuals and forecasts (same length) and returns MAE, MAPE, RMSE, and count. Here we align two quarters of revenue actuals vs the baseline forecast.


In [ ]:
from finstack_quant.statements_analytics import backtest_forecast

actual = [102.0, 108.0]
forecast = [100.0, 110.0]
bt = backtest_forecast(actual, forecast)
print(f"MAE={bt.mae:.2f}  MAPE={bt.mape:.2f}%  RMSE={bt.rmse:.2f}  n={bt.n}")

## Additional introspection APIs

Beyond `DependencyTracer` and `explain_formula`, the analytics module provides:

- **`DependencyTracer.dependency_tree_detailed`** — ASCII tree annotated with values for a specific period
- **`DependencyTracer.all_dependencies`** — flat list of all transitive upstream nodes
- **`DependencyTracer.dependents`** — reverse lookup: which nodes depend on a given node?
- **`explain_formula(...).to_text()`** — human-readable multi-line explanation

In [ ]:
from finstack_quant.statements_analytics import DependencyTracer, explain_formula

tracer = DependencyTracer(spec)
print("=== DependencyTracer.dependency_tree_detailed (ebitda, 2025Q1) ===")
print(tracer.dependency_tree_detailed(result, "ebitda", "2025Q1"))

print("=== all_dependencies(ebitda) ===")
print(tracer.all_dependencies("ebitda"))

print("\n=== dependents(revenue) — who uses revenue? ===")
print(tracer.dependents("revenue"))

print("\n=== explain_formula(ebitda, 2025Q1).to_text() ===")
print(explain_formula(spec, result, "ebitda", "2025Q1").to_text())

### Pandas export

`StatementResult` exports to pandas DataFrames in both long and wide forms, matching the notebook workflows shown in the statement modeling notebook.

In [ ]:
print("Pandas wide export:")
print(result.to_pandas(orient="wide"))
print()
print("Pandas long export (first 8 rows):")
print(result.to_pandas(orient="long").head(8))

## Mini-example: full analytics pass on the demo P&L

The cell below is a recap: it reuses the baseline `spec` / `result` from the shared-baseline cell and the `comp_result` upside plan from the variance section, then runs sensitivity, variance, scenarios, goal seek, dependency tracing, formula explanation, and a tiny revenue backtest back-to-back — mirroring how you might script a quarterly analytics pack. Run the notebook top to bottom so those objects exist.


In [ ]:
# Reuses the already-built objects: `spec` / `result` (baseline)
# and `comp_result` (upside plan, built in the variance section).

print("=== Sensitivity (EBITDA tornado, 2025Q3) ===")
sens_cfg = SensitivityConfig(
    "diagonal",
    [("revenue", "2025Q3", 115.0, [95.0, 105.0, 110.0, 115.0, 120.0, 125.0, 135.0])],
    ["gross_profit", "ebitda"],
)
sens = run_sensitivity(spec, sens_cfg)
tornado_entries = generate_tornado_entries(sens, "ebitda", "2025Q3")
print("Tornado:", [
    {"parameter_id": entry.parameter_id, "downside": entry.downside, "upside": entry.upside}
    for entry in tornado_entries
])

print("\n=== Variance (Q1–Q2, gross_profit & ebitda) ===")
var_cfg = VarianceConfig(
    "base",
    "upside",
    ["gross_profit", "ebitda"],
    ["2025Q1", "2025Q2"],
)
var_result = run_variance(result, comp_result, var_cfg)
for row in var_result.rows:
    print(f"  {row.period} {row.metric:15s}  Δ={row.abs_var:+.1f} ({row.pct_var:+.1%})")

print("\n=== Scenarios (revenue shocks, forecast periods only) ===")
scen = ScenarioSet({
    "upside": {"revenue": 130.0},
    "downside": {"revenue": 100.0},
})
scen_out = evaluate_scenario_set(spec, scen)
for name in scen_out.names:
    scenario_result = scen_out.get(name)
    print(
        f"  {name}: EBITDA Q3={scenario_result.get('ebitda', '2025Q3')}, Q4={scenario_result.get('ebitda', '2025Q4')}"
    )

print("\n=== Goal seek EBITDA 2025Q3 -> 25 ===")
gs = goal_seek(spec, goal_spec)
print("Solved revenue:", gs.solved_value)

print("\n=== Dependencies for ebitda ===")
print(DependencyTracer(spec).dependency_tree_text("ebitda"))
print("Direct:", DependencyTracer(spec).direct_dependencies("ebitda"))
print("Explain:", json.dumps(json.loads(explain_formula(spec, result, "ebitda", "2025Q1").to_json()), indent=2))

print("\n=== Revenue forecast backtest (2 quarters) ===")
bt = backtest_forecast([102.0, 108.0], [100.0, 110.0])
print(f"MAE={bt.mae:.2f}  MAPE={bt.mape:.2f}%  RMSE={bt.rmse:.2f}  n={bt.n}")


## Monte Carlo simulation

`Evaluator.evaluate_monte_carlo` runs paths using the model's forecast distributions (normal/lognormal etc.) and returns percentiles + optional path data.


In [ ]:
from finstack_quant.statements import Evaluator, MonteCarloConfig

mc_cfg = MonteCarloConfig(1000, 42, ["ebitda"], [0.05, 0.5, 0.95])
mc = Evaluator().evaluate_monte_carlo(spec, mc_cfg)
print("MC paths:", mc.n_paths)
print("Forecast periods:", mc.forecast_periods)
print("EBITDA p50:", mc.percentile_by_period("ebitda", 0.5))

## Credit assessment (dict form)

`credit_assessment` returns a structured dict (vs the text `credit_assessment_report_text`).


In [ ]:
from finstack_quant.statements_analytics import credit_assessment

ca = credit_assessment(result, "2025Q4")
print("credit_assessment:", ca.period, "leverage=", ca.leverage_ratio, "coverage=", ca.interest_coverage)

## Checks framework

The `CheckSuiteSpec` / `run_checks*` + renderers provide structured validation (three-statement, credit underwriting, custom). They are part of the surface but the primary notebook demos for checks live in dedicated model notebooks (covenant_monitoring) or are exercised via JSON configs in other tiers. See `covenant_monitoring.ipynb` and the analytics module docs for run_three_statement_checks / render_check_report_text usage patterns.


In [ ]:
import json as _json

from finstack_quant import statements as _st
from finstack_quant.statements_analytics import (
    run_checks,
    render_check_report_text,
    render_check_report_html,
)

# A model to validate
_cb = _st.ModelBuilder("checks_demo")
_cb.periods("2024Q1..Q2", None)
_cb.value("revenue", [("2024Q1", 100_000.0), ("2024Q2", 110_000.0)])
_cb.value("cogs", [("2024Q1", 40_000.0), ("2024Q2", 44_000.0)])
_cb.compute("gross_profit", "revenue - cogs")
_check_model = _cb.build()

# A check suite: parse it into a typed CheckSuiteSpec, then run it
suite_json = _json.dumps({
    "name": "P&L sanity suite",
    "builtin_checks": [],
    "formula_checks": [
        {
            "id": "revenue_positive",
            "name": "Revenue must be positive",
            "category": "internal_consistency",
            "severity": "error",
            "mode": "predicate",
            "scope": "all_periods",
            "formula": "revenue > 0",
            "message_template": "Revenue not positive in {period}",
        }
    ],
})
suite = _st.CheckSuiteSpec.from_json(suite_json)
print(f"CheckSuiteSpec '{suite.name}': {suite.formula_check_count} formula check(s)")

report = run_checks(_check_model, suite)
print(
    f"CheckReport: passed={report.passed} checks={report.total_checks} errors={report.total_errors} warnings={report.total_warnings}"
)
print("\n--- text render ---")
print(render_check_report_text(report))
print("html render length:", len(render_check_report_html(report)))

# Mapping-driven check suites: map model nodes to standard statement lines.
from finstack_quant.statements_analytics import (
    run_three_statement_checks,
    run_credit_underwriting_checks,
)

_mb = _st.ModelBuilder("ts_demo")
_mb.periods("2024Q1..Q2", None)
for node, vals in {
    "total_assets": [1000.0, 1100.0],
    "total_liabilities": [600.0, 650.0],
    "total_equity": [400.0, 450.0],
    "cash": [100.0, 130.0],
    "retained_earnings": [200.0, 250.0],
    "net_income": [60.0, 70.0],
    "interest_expense": [10.0, 11.0],
    "ebitda": [120.0, 130.0],
    "total_debt": [500.0, 520.0],
}.items():
    _mb.value(node, [("2024Q1", vals[0]), ("2024Q2", vals[1])])
_ts_model = _mb.build()

ts_mapping = {
    "assets_nodes": ["total_assets"],
    "liabilities_nodes": ["total_liabilities"],
    "equity_nodes": ["total_equity"],
    "cash_node": "cash",
    "retained_earnings_node": "retained_earnings",
    "net_income_node": "net_income",
    "interest_expense_node": "interest_expense",
}
ts_report = run_three_statement_checks(_ts_model, ts_mapping)
print(f"\nrun_three_statement_checks: checks={ts_report.total_checks} passed={ts_report.passed}")

credit_mapping = {
    "debt_node": "total_debt",
    "ebitda_node": "ebitda",
    "interest_expense_node": "interest_expense",
    "cash_node": "cash",
}
cu_report = run_credit_underwriting_checks(_ts_model, credit_mapping)
print(f"run_credit_underwriting_checks: checks={cu_report.total_checks} passed={cu_report.passed}")

## Takeaways

- **Sensitivity** (`run_sensitivity`, `generate_tornado_entries`) quantifies how driver shocks flow into key metrics before you commit to a plan.
- **Variance** (`run_variance`) formalizes baseline vs alternate evaluated results—ideal for budget vs actual or plan versions.
- **Goal seek** finds the driver level that hits a target metric in a period, preserving the rest of the model structure.
- **Scenarios** (`evaluate_scenario_set`) batch-evaluate named override sets; overrides are node-level scalars applied only to **forecast periods** (periods after `actuals_until`).
- **Introspection** (`DependencyTracer.dependency_tree`, `DependencyTracer.direct_dependencies`, `explain_formula`) supports audit trails and IC-ready narrative.
- **Backtesting** (`backtest_forecast`) scores vector forecasts with standard error metrics.
- **Export** to **pandas** with `to_pandas(orient="wide")` and `to_pandas(orient="long")`.

**Next:** Keep building richer statement models in `04_statement_modeling/statement_modeling.ipynb`, then wire these analytics into your own reporting or scenario libraries.
